# Hyperopt — Hyperparameter Optimization for XGBoost

In [ ]:
# Import libraries for optimisation and modelling
import numpy as np
import pandas as pd
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import xgboost as xgb

## 1. Load & Prepare Data

In [ ]:
# Load cleaned sales data and create simple lag/rolling features
df = pd.read_csv("data/cleaned_timeseries.csv", parse_dates=["date"])
df = df.sort_values("date").set_index("date").asfreq("D")

df["lag_7"] = df["unit_sales"].shift(7)
df["lag_14"] = df["unit_sales"].shift(14)
df["rolling_mean_7"] = df["unit_sales"].rolling(7, min_periods=1).mean()
df["rolling_mean_30"] = df["unit_sales"].rolling(30, min_periods=1).mean()

# Drop rows with NaN from lag features
df = df.dropna()

# Train/test split (same period as other notebooks)
train = df.loc["2013-01-01":"2013-12-31"]
test = df.loc["2014-01-01":"2014-03-31"]

features = ["lag_7", "lag_14", "rolling_mean_7", "rolling_mean_30"]
X_train, y_train = train[features], train["unit_sales"]
X_test, y_test = test[features], test["unit_sales"]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Define Search Space

In [ ]:
# Hyperparameter ranges for XGBoost to explore
space = {
    'max_depth':        hp.quniform('max_depth', 3, 10, 1),
    'learning_rate':    hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
    'n_estimators':     hp.quniform('n_estimators', 50, 300, 10),
    'subsample':        hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'reg_lambda':       hp.uniform('reg_lambda', 0.1, 2.0),
    'reg_alpha':        hp.uniform('reg_alpha', 0.0, 1.0),
}

## 3. Objective Function

In [ ]:
# Objective: train XGBoost with given params, return RMSE (minimised)
def objective(params):
    params = {
        'max_depth':        int(params['max_depth']),
        'learning_rate':    params['learning_rate'],
        'n_estimators':     int(params['n_estimators']),
        'subsample':        params['subsample'],
        'colsample_bytree': params['colsample_bytree'],
        'reg_lambda':       params['reg_lambda'],
        'reg_alpha':        params['reg_alpha'],
        'random_state':     42,
        'n_jobs':           -1,
        'objective':        'reg:squarederror'
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    return {'loss': rmse, 'status': STATUS_OK}

## 4. Run Hyperopt

In [ ]:
# Run Tree-structured Parzen Estimator (TPE) for 50 evaluations
trials = Trials()
best = fmin(fn=objective, space=space, algo=tpe.suggest,
            max_evals=50, trials=trials)

print("\nBest hyperparameters:")
print(best)

## 5. Train Final Model with Best Params

In [ ]:
# Convert best params to proper types and evaluate on test set
best_params = {k: int(v) if k in ('max_depth','n_estimators') else v
               for k, v in best.items()}
best_params.update({'random_state': 42, 'n_jobs': -1, 'objective': 'reg:squarederror'})

final = xgb.XGBRegressor(**best_params)
final.fit(X_train, y_train)

train_rmse = np.sqrt(mean_squared_error(y_train, final.predict(X_train)))
test_rmse = np.sqrt(mean_squared_error(y_test, final.predict(X_test)))
print(f"Train RMSE: {train_rmse:.1f}")
print(f"Test RMSE:  {test_rmse:.1f}")